# Silver → Gold: dim_delegate

**Propósito:** Crear la dimensión de delegados/comerciales extrayéndola de `dbo.salestrack_sales_final` en Silver. No existe como tabla propia en Bronze.

**Lógica:** DISTINCT de `delegate_id` + `delegate_name`. En caso de que un mismo `delegate_id` tenga varios nombres (datos sucios), se toma el más frecuente.

**Idempotencia:** `overwrite` + `overwriteSchema=true`.

In [6]:
%run ./config

StatementMeta(, 363f978d-fb14-49fa-b009-8254f5d9ff19, 9, Finished, Available, Finished, True)

Config cargada → Bronze: lakehouse_poc | Silver: silver_lakehouse_poc | Gold: gold_lakehouse_poc


In [7]:


SILVER_TABLE = f"{DEFAULT_SCHEMA}.salestrack_sales_final"
GOLD_TABLE   = f"{DEFAULT_SCHEMA}.dim_delegate"

StatementMeta(, 363f978d-fb14-49fa-b009-8254f5d9ff19, 10, Finished, Available, Finished, False)

In [8]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime
from pyspark.sql.window import Window

df_silver = spark.read.table(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")

print(f"Filas fuente (salestrack): {df_silver.count()}")

StatementMeta(, 363f978d-fb14-49fa-b009-8254f5d9ff19, 11, Finished, Available, Finished, False)

Filas fuente (salestrack): 261239


In [9]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────
# Si un delegate_id tiene varios nombres, tomamos el más frecuente
w = Window.partitionBy("delegate_id").orderBy(F.col("freq").desc())

df_gold = (
    df_silver
    .filter(F.col("delegate_id").isNotNull())
    .select("delegate_id", "delegate_name")
    .groupBy("delegate_id", "delegate_name")
    .agg(F.count("*").alias("freq"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") == 1)
    .drop("freq", "rank")
    .withColumn("delegate_name", F.trim(F.col("delegate_name")))
    .withColumn("gold_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
)

StatementMeta(, 363f978d-fb14-49fa-b009-8254f5d9ff19, 12, Finished, Available, Finished, False)

In [10]:
# ─── VALIDACIÓN ───────────────────────────────────────────────────────────────
row_count = df_gold.count()
null_ids  = df_gold.filter(F.col("delegate_id").isNull()).count()
dup_ids   = df_gold.groupBy("delegate_id").count().filter(F.col("count") > 1).count()

print(f"Delegados únicos en Gold  : {row_count}")
print(f"delegate_id nulos         : {null_ids}")
print(f"delegate_id duplicados    : {dup_ids}")

assert null_ids == 0, "ERROR: hay delegate_id nulos"
assert dup_ids  == 0, "ERROR: hay delegate_id duplicados tras deduplicación"

df_gold.show()

StatementMeta(, 363f978d-fb14-49fa-b009-8254f5d9ff19, 13, Finished, Available, Finished, False)

Delegados únicos en Gold  : 115
delegate_id nulos         : 0
delegate_id duplicados    : 0
+-----------+-----------------+--------------------+
|delegate_id|    delegate_name|        gold_load_ts|
+-----------+-----------------+--------------------+
|      24997|  María Rodríguez|2026-04-22 11:29:...|
|      31700|  Catalina Chacón|2026-04-22 11:29:...|
|      31701|   Marifé Vázquez|2026-04-22 11:29:...|
|      31703|     Rosa Pasamón|2026-04-22 11:29:...|
|      34734|     María Juárez|2026-04-22 11:29:...|
|      40228|Estefanía Grazini|2026-04-22 11:29:...|
|      40233|   Míriam Sánchez|2026-04-22 11:29:...|
|      40237|   Marta Neubauer|2026-04-22 11:29:...|
|      40238|     Angel Puebla|2026-04-22 11:29:...|
|      40240|  Beatriz Perales|2026-04-22 11:29:...|
|      40247|  Irene Contreras|2026-04-22 11:29:...|
|      40248|   Beatriz Arranz|2026-04-22 11:29:...|
|      40328|  Yolanda Reguera|2026-04-22 11:29:...|
|      40329|   Almudena Torre|2026-04-22 11:29:...|
|      

In [11]:
# ─── ESCRITURA IDEMPOTENTE EN GOLD ────────────────────────────────────────────
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{GOLD_LAKEHOUSE}`.{GOLD_TABLE}")
)

print(f"✓ Tabla {GOLD_LAKEHOUSE}.{GOLD_TABLE} escrita correctamente con {row_count} filas.")

StatementMeta(, 363f978d-fb14-49fa-b009-8254f5d9ff19, 14, Finished, Available, Finished, True)

✓ Tabla gold_lakehouse_poc.dbo.dim_delegate escrita correctamente con 115 filas.
